Today's topics:
* NumPy arrays and elementwise arithmetic
* broadcasting and shape errors
* indexing into an array, forward and from the end

# A gap in the periodic table

In 1869, Dmitri Mendeleev arranged the known elements by atomic weight and found that their properties repeated on a cycle.
To keep the cycle lined up, he had to leave some positions empty, and wrote a question mark where an element should have been.

One sat just below silicon, written that first year as `? = 70`. Mendeleev
called the missing element eka-silicon, and over the next two years he sharpened
the estimate from the position alone: atomic weight near 72, density near
5.5 g/cm$^3$. Nobody had ever seen it.
([Royal Society of Chemistry: Germanium](https://periodic-table.rsc.org/element/32/germanium))

In 1886 Clemens Winkler isolated a new element and measured it. We're going to
look up what he found, in a file holding all 118 elements.

We'll get there in about twenty minutes.
First we need the data structure to hold 118 elements.

# Working with batches of data

Last class we computed density for five hypothetical samples. A real tensile test might
pull eight coupons. A hardness map might press a hundred indents across a
weld. The periodic table has 118 elements.

You could make a variable for every measurement, but ten samples means
twenty variables and the density calculation written out ten times. We need
one variable that holds many numbers in order (hundreds, thousands, millions!).

## Modules

So far Python has given us variables, arithmetic, types, and printing. That's
less than a graphing calculator can do. The reason Python does more is
**modules**: code other people wrote and gave away, which you load into your own
program with the `import` keyword.

[NumPy](https://numpy.org/) is the standard module for numerical computing in
Python. Almost any program with equations or data in it imports NumPy.

In [1]:
import numpy
print(numpy.pi)

3.141592653589793


`import` prints nothing on its own unless something goes wrong. The name
`numpy` now holds an entire library, and everything in it is reached by
writing the module name, a dot, and the thing you want.

## Aliases

A working program types the module name hundreds of times, so NumPy is almost
always renamed on the way in. The `as` keyword does that:

In [2]:
import numpy as np
print(np.pi)

3.141592653589793


`np` is a name that points at the NumPy module, the same way `mass_g` pointed
at `44.5` last week. Everything NumPy offers is reached through it, written
`np.something`. You'll see `import numpy as np` at the top of nearly every
scientific Python file you ever read.

## Creating arrays with NumPy

To make an array, write `np.array([...])` with your values inside the square
brackets, separated by commas.

Here is a shipment of cast aluminum-alloy bars, ten samples (synthetic
teaching data). The first five are the same ones from last class. Mass in
grams:

In [3]:
bar_mass = np.array([27.0, 41.6, 13.5, 54.2, 20.8, 33.9, 47.3, 16.9, 60.7, 24.5])
print(bar_mass)

[27.  41.6 13.5 54.2 20.8 33.9 47.3 16.9 60.7 24.5]


And the volume of each, now in cm$^3$ instead of mL.
The variables picked up a `bar_` prefix so we know what they belong to.

In [4]:
bar_volume = np.array([10.0, 15.0, 5.0, 20.0, 8.0, 12.0, 17.5, 6.5, 22.0, 9.0])
print(bar_volume)

[10.  15.   5.  20.   8.  12.  17.5  6.5 22.   9. ]


We made one variable, but it's holding ten numbers. Let's check what kind of
object we made and how many entries it holds:

In [5]:
print(type(bar_mass))
print(len(bar_mass))

<class 'numpy.ndarray'>
10


The `ndarray` part is short for "n-dimensional array."

`bar_mass` and `bar_volume` are both 10 entries long. Entry `i` of one and
entry `i` of the other are the same physical sample. Two arrays used this way
are called **parallel**, and their shared ordering is part of the data structure.

# Elementwise arithmetic

The operators from last class work on arrays. NumPy applies them entry by entry,
which is called **elementwise**:

In [6]:
bar_density = bar_mass / bar_volume
print(bar_density)

[2.7        2.77333333 2.7        2.71       2.6        2.825
 2.70285714 2.6        2.75909091 2.72222222]


That one line computed all ten densities: first mass divided by first volume,
second mass divided by second volume, and so on. If the shipment had a
thousand bars, the line would look exactly the same.

## Broadcasting

Suppose the balance reads 0.5 g too low (a tare error), so every mass needs the same
correction:

In [7]:
corrected_bar_mass = bar_mass + 0.5
print(corrected_bar_mass)

[27.5 42.1 14.  54.7 21.3 34.4 47.8 17.4 61.2 25. ]


NumPy applied `0.5` to all ten entries. This operation is called
**broadcasting**. NumPy stretches the single number to match the array so
every entry gets a partner. The same pattern works with `-`, `*`, `/`,
and `**`.

In [8]:
print(bar_mass - 2.0)
print(bar_mass / 1000)    # g to kg

[25.  39.6 11.5 52.2 18.8 31.9 45.3 14.9 58.7 22.5]
[0.027  0.0416 0.0135 0.0542 0.0208 0.0339 0.0473 0.0169 0.0607 0.0245]


## When shapes don't match

Broadcasting can't invent a pairing when the lengths are incompatible:

In [9]:
# EXPECTED-ERROR: this cell fails on purpose: see the surrounding text
wrong_size = np.array([1.0, 2.0, 3.0])
print(bar_mass + wrong_size)

ValueError: operands could not be broadcast together with shapes (10,) (3,) 

Read the last line first. NumPy could not broadcast shapes `(10,)` and `(3,)`.
`(10,)` means ten entries along one axis. Ten and three don't match, so NumPy
stops. Check `len()` when you see this error.

In [10]:
print(len(bar_mass), len(wrong_size))

10 3


Matching lengths still wouldn't prove that the specimens or the units were
paired correctly. NumPy checks the shapes, not whether the data lines up the
way you intended.

## Guided practice: screening a batch

Pure aluminum is about 2.70 g/cm$^3$. Alloying shifts that a little, but a value
far from 2.70 means a mislabeled sample or a measurement error, not a new alloy.

Let's check the range.

In [11]:
print(f'lowest:  {np.min(bar_density):.2f} g/cm^3')
print(f'highest: {np.max(bar_density):.2f} g/cm^3')

lowest:  2.60 g/cm^3
highest: 2.82 g/cm^3


2.60 to 2.83 g/cm$^3$. All ten are plausible for aluminum alloys, so nothing
needs to be pulled and re-measured.

### [Check your understanding]

A tensile test on 8 coupons recorded the force at which each one yielded, and
the cross-sectional area of each coupon:

```
applied_force      = [4200, 5600, 3100, 7400, 6000, 4800, 8900, 5300]   # N
cross_section_area = [20.0, 25.0, 15.5, 30.0, 28.0, 22.0, 35.0, 24.0]   # mm^2
```

Stress is force divided by area.
Force in newtons over area in mm$^2$ gives MPa directly.

In the code cell below:

1. Build both of these as NumPy arrays
2. Compute `stress = applied_force / cross_section_area`
3. Convert to ksi by multiplying by 0.145. One ksi is 1000 psi, and
   1 MPa is 0.145 ksi. You'll often see ksi on US mill certificates and in
   handbooks. This is broadcasting again: one conversion factor
   applied to the whole array.
4. Print the ksi values and the largest one
5. Also print the smallest stress and the range (largest minus smallest), in ksi

*Optional challenge*

6. A second machine reads 3% high across the whole batch. Divide your stress
   array by 1.03 to correct it, then print the corrected largest value. This is
   broadcasting again.

## What the screening actually tells you

Receiving labs compare measured densities with handbook values before putting
material into production. A low density can indicate porosity from casting
defects. A high density can indicate a mislabeled alloy. Either way, the
batch gets flagged for more testing.

# Back to the table

The periodic table is a batch of 118. The cell below loads five parallel
arrays, ordered by atomic number.

In [12]:
#@title Load the real periodic-table dataset (click ▶ to run, data loading, not a learning objective) { display-mode: "form" }
import os

import numpy as np
import pandas as pd

_file = 'mendeleev_elements.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}',
                 f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    _table = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e

symbol = _table['symbol'].to_numpy()
atomic_number = _table['atomic_number'].to_numpy()
radius_pm = _table['covalent_radius_cordero'].to_numpy()
density = _table['density'].to_numpy()
atomic_weight = _table['atomic_weight'].to_numpy()

print(f'Loaded {len(symbol)} elements from {_file}')

Loaded 118 elements from mendeleev_elements.csv


In [13]:
print(type(symbol), len(symbol))
print(type(atomic_weight), len(atomic_weight))

<class 'numpy.ndarray'> 118
<class 'numpy.ndarray'> 118


The cell created five parallel arrays with 118 entries each. Position `i`
refers to the same element in every array. You don't need to know how the
loading code works yet; for now, let's use the arrays it gave us.

Germanium is element 32:

In [14]:
print(symbol[31])
print(atomic_weight[31])
print(density[31])

Ge
72.63
5.3234


Germanium has an atomic weight of 72.63 and a density of 5.3234 g/cm$^3$.

Let's put Mendeleev's predictions next to what Winkler actually measured:

| Property | Mendeleev predicted (1871) | Winkler measured (1886) |
|---|---|---|
| Atomic weight | ~72 | 72.63 |
| Density (g/cm$^3$) | ~5.5 | 5.32 |

He was within 3% on both, from an empty box in the table.

Germanium is element **32**, but we found it at index **31**.

*Why does index 31 find element 32?*

# Zero-based indexing

Let's talk about the `0` here. It might make more sense to call the first
entry "1," but Python calls it `0`. The first entry of an array is
`array[0]`, not `array[1]`.

In [15]:
print(symbol[0])
print(symbol[1])
print(symbol[5])

H
He
C


So `symbol[0]` is hydrogen, element 1, and `symbol[5]` is carbon, element 6.
The array is ordered by atomic number, so index = atomic number - 1:

In [16]:
print(atomic_number[31])
print(atomic_number[5])

32
6


The pattern is that `array[i]` gives you item number `i + 1`. Because Python
starts at zero, the valid indices run from `0` through `len(array) - 1`.

In [17]:
print(len(symbol))

118


118 elements, so the valid indices are 0 through 117.

*What happens if we ask for one that isn't there?*

In [18]:
# EXPECTED-ERROR: this cell fails on purpose: see the surrounding text
print(symbol[118])

IndexError: index 118 is out of bounds for axis 0 with size 118

`IndexError: index 118 is out of bounds`. Remember the strategy from last class:
read the last line first. There's no element 119, and no index 118 either.

## Negative indexing

A very useful trick is to reference the last entry with `-1`. You can think
of it as starting at the first entry and taking one step backward to wrap
around to the end. Then `-2` gives you the second-to-last entry:

In [19]:
print(symbol[-1])
print(symbol[-2])

Og
Ts


`symbol[-1]` returns oganesson, element 118. Let's confirm that `-1` and
`117` give the same entry.

In [20]:
print(symbol[117])
print(symbol[-1])

Og
Og


Both indices return the same element. I like `-1` because it still works when
you don't know how long the array is.

### [Check your understanding]

Using the periodic-table arrays, write code that:

1. Prints the symbol, atomic weight, and density of oxygen, element 8.
   Remember index = atomic number - 1.
2. Prints the symbol of the last element in the table using a negative index.
3. Prints the symbol of the second-to-last element two ways: once with a
   negative index, and once with a positive index you compute from
   `len(symbol)`. Confirm they match.

*Optional challenge*

4. The array is ordered so `atomic_number[i]` equals `i + 1`. Pick any element,
   print its `atomic_number` entry, and confirm it matches the position you used
   to look it up.

## Position is data

All of this works because entry `i` of every array refers to the same thing.
Index 31 meant germanium in `symbol`, `density`, and `atomic_weight`.

Nothing in numpy enforces that. If someone handed you a radius array sorted
alphabetically and you indexed into it with an atomic number, every answer
would be wrong and nothing would crash.

Index carries meaning only because you keep the ordering convention.

## Further reading

- [VanderPlas: Introduction to NumPy](https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html)
- [VanderPlas: The basics of NumPy arrays](https://jakevdp.github.io/PythonDataScienceHandbook/02.02-the-basics-of-numpy-arrays.html)
- [McKinney: NumPy basics](https://wesmckinney.com/book/numpy-basics)
- [Cordero et al., "Covalent radii revisited" (2008)](https://doi.org/10.1039/B801115J), the source of the radii used today